In [1]:
import pandas as pd
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score



In [7]:
# load dataset
v1 = pd.read_csv("hf://datasets/AnkitAI/product-reviews-sentiment/categorized_text_reviews.csv")
v2 = pd.read_csv("hf://datasets/AnkitAI/product-reviews-sentiment/synthetic_reviews.csv")
data = pd.concat([v1, v2], ignore_index=True)

print("datast loaded:", len(data))
print("Columns:", data.columns.tolist())
print("\nfirst 3 rows:")
print(data.head(3))

datast loaded: 9000
Columns: ['review', 'category']

first 3 rows:
                                              review          category
0  The product's innovative features have greatly...  Product Feedback
1  Customer service was unresponsive to my inquir...  Customer Service
2  I was a victim of fraud, the seller requested ...    Fraud and Scam


In [3]:
# Map categories to sentiment
def get_sentiment(category):
    if category in ['Operational Issues', 'Fraud and Scam']:
        return 'negative'
    else:
        return 'positive'

data['sentiment'] = data['category'].apply(get_sentiment)

print("Sentiment distribution:")
print(data['sentiment'].value_counts())
print("\nSample data:")
print(data[['review', 'category', 'sentiment']].head(5))

Sentiment distribution:
sentiment
negative    4555
positive    4445
Name: count, dtype: int64

Sample data:
                                              review            category  \
0  The product's innovative features have greatly...    Product Feedback   
1  Customer service was unresponsive to my inquir...    Customer Service   
2  I was a victim of fraud, the seller requested ...      Fraud and Scam   
3  The delivery was delayed due to unforeseen cir...  Operational Issues   
4  The product arrived with some defects, but the...    Product Feedback   

  sentiment  
0  positive  
1  positive  
2  negative  
3  negative  
4  positive  


In [4]:
# Prepare data
X = data['review']
y = data['sentiment']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train model
model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
    ('classifier', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

print(" Model trained successfully!")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

 Model trained successfully!
Training samples: 7200
Testing samples: 1800


In [5]:
# Predict and evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)
print("\nFirst 10 predictions vs actual:")
for i in range(10):
    print(f"Predicted: {y_pred[i]:8} | Actual: {y_test.iloc[i]:8} | Review: {X_test.iloc[i][:50]}...")

Model Accuracy: 0.9844444444444445

First 10 predictions vs actual:
Predicted: positive | Actual: positive | Review: Ordered a pair of hiking boots for my husband. The...
Predicted: positive | Actual: positive | Review: Customer service was helpful and provided a quick ...
Predicted: negative | Actual: negative | Review: No signs of fraudulent activity or scam during my ...
Predicted: negative | Actual: negative | Review: This site is a scam. They took my money and never ...
Predicted: negative | Actual: negative | Review: Someone hacked my account on a toy resale site and...
Predicted: negative | Actual: negative | Review: Pre-ordered the new gaming headset weeks ago, rele...
Predicted: negative | Actual: negative | Review: There were glitches on the website, making it diff...
Predicted: positive | Actual: positive | Review: I ordered sandals for my mom and they didn't fit, ...
Predicted: negative | Actual: negative | Review: There were glitches on the website, making it diff...
Predi

In [8]:
test_feedback = [
    "Payment gateway is not working",
    "The application is very slow",
    "I love this app, it's amazing",
    "Customer support was very helpful",
    "App crashes every time I try to login"
]

predictions = model.predict(test_feedback)

print("sentiment Analysis Results:")
print("-"*50)
for text, sentiment in zip(test_feedback, predictions):
    print(f"{text}")
    print(f"sentiment: {sentiment.upper()}")
    print("-"*50)

sentiment Analysis Results:
--------------------------------------------------
Payment gateway is not working
sentiment: POSITIVE
--------------------------------------------------
The application is very slow
sentiment: POSITIVE
--------------------------------------------------
I love this app, it's amazing
sentiment: POSITIVE
--------------------------------------------------
Customer support was very helpful
sentiment: POSITIVE
--------------------------------------------------
App crashes every time I try to login
sentiment: NEGATIVE
--------------------------------------------------


In [9]:
analyzer=SentimentIntensityAnalyzer()
negative_keywords = ['not working', 'slow', 'laggy', 'crash', 'failed', 'failing', 
                    'terrible', 'awful', 'horrible', 'useless', 'scam', 'fraud']

positive_keywords = ['love', 'amazing', 'excellent', 'awesome', 'great', 'perfect',
                    'wonderful', 'fantastic', 'helpful', 'quickly', 'solved']
def hybrid_sentiment(text):
    text_lower=text.lower()
    neg_count = sum(1 for word in negative_keywords if word in text_lower)
    pos_count = sum(1 for word in positive_keywords if word in text_lower)
    score=analyzer.polarity_scores(text)
    if neg_count > pos_count:
        return 'negative'
    elif pos_count > neg_count:
        return 'positive'
    else:
        if scores['compound'] >= 0.05:
            return 'positive'
        elif scores['compound'] <= -0.05:
            return 'negative'
        else:
            return 'neutral'
    

In [11]:
test_feedback = [
    "Payment gateway is not working",
    "The application is very slow",
    "I love this app, it's amazing",
    "Customer support was very helpful",
    "App crashes every time I try to login",
    "The product is terrible"
]
print("hybrid sentiment analysis")
print("-"*50)
for text in test_feedback:
    sentiment=hybrid_sentiment(text)
    print(f"{text}")
    print(f"sentiment :{sentiment.upper()}")
    print("-"*50)

hybrid sentiment analysis
--------------------------------------------------
Payment gateway is not working
sentiment :NEGATIVE
--------------------------------------------------
The application is very slow
sentiment :NEGATIVE
--------------------------------------------------
I love this app, it's amazing
sentiment :POSITIVE
--------------------------------------------------
Customer support was very helpful
sentiment :POSITIVE
--------------------------------------------------
App crashes every time I try to login
sentiment :NEGATIVE
--------------------------------------------------
The product is terrible
sentiment :NEGATIVE
--------------------------------------------------


In [12]:
def analyze_feedback(text):
    """Complete feedback analysis: sentiment + category + keywords"""
    
    # 1. Sentiment
    text_lower = text.lower()
    neg_count = sum(1 for word in negative_keywords if word in text_lower)
    pos_count = sum(1 for word in positive_keywords if word in text_lower)
    scores = analyzer.polarity_scores(text)
    
    if neg_count > pos_count:
        sentiment = 'negative'
    elif pos_count > neg_count:
        sentiment = 'positive'
    else:
        if scores['compound'] >= 0.05:
            sentiment = 'positive'
        elif scores['compound'] <= -0.05:
            sentiment = 'negative'
        else:
            sentiment = 'neutral'
    
    # 2. Category
    categories = []
    if 'payment' in text_lower or 'pay' in text_lower:
        categories.append('Payment')
    if 'slow' in text_lower or 'lag' in text_lower or 'crash' in text_lower:
        categories.append('Performance')
    if 'login' in text_lower or 'otp' in text_lower or 'password' in text_lower:
        categories.append('Login')
    if 'support' in text_lower or 'help' in text_lower:
        categories.append('Support')
    if not categories:
        categories.append('General')
    
    # 3. Keywords
    words = text_lower.split()
    keywords = [w for w in words if len(w) > 3 and w not in ['this', 'that', 'with', 'from', 'have']]
    keywords = keywords[:4]
    
    return {
        'sentiment': sentiment,
        'categories': categories,
        'keywords': keywords
    }


In [13]:
test_feedback = [
    "Payment gateway is not working",
    "I love this app, it's amazing",
    "App crashes every time I try to login",
    "Customer support was very helpful",
    "Payment failed and app is slow"
]

print("=" * 60)
print("FINAL CUSTOMER FEEDBACK ANALYSIS SYSTEM")
print("=" * 60)

for text in test_feedback:
    result = analyze_feedback(text)
    print(f"\n {text}")
    print(f"Sentiment: {result['sentiment'].upper()}")
    print(f"Categories: {', '.join(result['categories'])}")
    print(f" Keywords: {', '.join(result['keywords'])}")
    print("-"*60)

FINAL CUSTOMER FEEDBACK ANALYSIS SYSTEM

 Payment gateway is not working
Sentiment: NEGATIVE
Categories: Payment
 Keywords: payment, gateway, working
------------------------------------------------------------

 I love this app, it's amazing
Sentiment: POSITIVE
Categories: General
 Keywords: love, app,, it's, amazing
------------------------------------------------------------

 App crashes every time I try to login
Sentiment: NEGATIVE
Categories: Performance, Login
 Keywords: crashes, every, time, login
------------------------------------------------------------

 Customer support was very helpful
Sentiment: POSITIVE
Categories: Support
 Keywords: customer, support, very, helpful
------------------------------------------------------------

 Payment failed and app is slow
Sentiment: NEGATIVE
Categories: Payment, Performance
 Keywords: payment, failed, slow
------------------------------------------------------------


In [14]:

my_feedback = [
    "The app is great but payment takes too long",
    "I cannot login, OTP is not coming to my phone",
    "The new dashboard design is beautiful",
    "Support team resolved my issue in 5 minutes"
]

for text in my_feedback:
    result = analyze_feedback(text)
    print(f"\n {text}")
    print(f" Sentiment: {result['sentiment'].upper()}")
    print(f" Categories: {', '.join(result['categories'])}")
    print(f" Keywords: {', '.join(result['keywords'])}")
    print("-"*60)


 The app is great but payment takes too long
 Sentiment: POSITIVE
 Categories: Payment
 Keywords: great, payment, takes, long
------------------------------------------------------------

 I cannot login, OTP is not coming to my phone
 Sentiment: NEUTRAL
 Categories: Login
 Keywords: cannot, login,, coming, phone
------------------------------------------------------------

 The new dashboard design is beautiful
 Sentiment: POSITIVE
 Categories: General
 Keywords: dashboard, design, beautiful
------------------------------------------------------------

 Support team resolved my issue in 5 minutes
 Sentiment: POSITIVE
 Categories: Support
 Keywords: support, team, resolved, issue
------------------------------------------------------------


In [15]:
from sklearn.metrics import classification_report,confusion_matrix
y_pred=model.predict(X_test)
print("="*60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred))

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
print(confusion_matrix(y_test, y_pred))


CLASSIFICATION REPORT
              precision    recall  f1-score   support

    negative       0.98      0.98      0.98       924
    positive       0.98      0.98      0.98       876

    accuracy                           0.98      1800
   macro avg       0.98      0.98      0.98      1800
weighted avg       0.98      0.98      0.98      1800


CONFUSION MATRIX
[[910  14]
 [ 14 862]]
